# New warp (PyTorch) — scratch from `warp_test_quantized` tabs 11–22

Torch translation of the downscale debug session in `warp_test_quantized.ipynb`:

* helper preamble: the framework stand-ins + torch versions of `generate_image`,
  `create_scale_grid`, `generate_relative_grid`, `grid_sample` and `infer_warp`
* work area: setup → grid dequant/prep → TRC mesh + corner clip → weights →
  corner fetch → BCU MAC → `trc()` stub → accumulator scale → golden comparison

Everything is torch tensors (`float64` math like the original numpy code,
`int16` grid, `long` indices). No numpy anywhere.

In [20]:
import os
import sys
# make npe_mx_emu importable whether jupyter runs from the repo root or from EL_nb/
_here = os.getcwd()
_cands = [_here, os.path.dirname(_here)]  # EL_nb/ -> parent
for _c in _cands:
    if os.path.isdir(os.path.join(_c, 'npe_mx_emu')) and _c not in sys.path:
        sys.path.insert(0, _c)
        break

import torch

from npe_mx_emu import MXObserver

torch.manual_seed(0)


# ---------------------------------------------------------------------------
# Minimal stand-ins for env.api.models classes (attribute holders only)
# ---------------------------------------------------------------------------
class Format:
    def __init__(self, sign=False, exponent=0, mantissa=8):
        self.sign = sign
        self.exponent = exponent
        self.mantissa = mantissa


class Dimensions:
    def __init__(self, x=0, y=0, z=0, n=0):
        self.x = x
        self.y = y
        self.z = z
        self.n = n


class NodeWarp:
    def __init__(self, limit=None, reference_offset=None):
        self.limit = limit
        self.reference_offset = reference_offset


class Node:
    def __init__(self, name='', warp=None):
        self.name = name
        self.warp = warp


class Blob:
    def __init__(self, name='', format=None):
        self.name = name
        self.format = format


# ---------------------------------------------------------------------------
# Copied from env/ir/inference_engine/infer_numpy/tester_utils.py
# ---------------------------------------------------------------------------
class MockGraph:
    class MockLog:
        def debug(self, msg):
            pass

    def __init__(self, mock_blob_data, blob_dict):
        self._mock_blob_data = mock_blob_data
        self._blob_dict = blob_dict
        self._log = MockGraph.MockLog()

    def get_node_blobs_data(self, node):
        return self._mock_blob_data

    def get_blob(self, node, blob_name):
        return self._blob_dict[blob_name]

    def set_blob_data(self, blob, blob_data):
        self._mock_blob_data[f'_{blob.name}'] = blob_data

    @property
    def log(self):
        return self._log

In [21]:
def generate_bg(width, height) -> torch.Tensor:
    grad = torch.round(torch.linspace(0, 1, height, dtype=torch.float64) * (255 - 135) + 135)
    val = torch.vstack([grad, torch.ones_like(grad) * 206, torch.ones_like(grad) * 235]).to(torch.uint8)
    sky = val.T.unsqueeze(1).expand(-1, width, -1).contiguous()   # (height, width, 3)
    return sky


def generate_green_ball(width, height, img) -> torch.Tensor:
    center = (width // 2, height // 2)  # Center of the image
    radius = width // 4
    xs, ys = torch.meshgrid((torch.arange(width) - center[0]).double(),
                            (torch.arange(height) - center[1]).double(), indexing='xy')
    dist = torch.hypot(xs, ys)
    mask = dist < radius
    grad = 1 - dist / radius
    ball = torch.zeros_like(img)
    ball[..., 1] = torch.clamp(grad * 255, 0, 255).to(img.dtype)
    img = torch.where(mask.unsqueeze(-1), ball, img)
    return img


def generate_image(height, width) -> torch.Tensor:

    # Create image
    image = generate_bg(width, height)

    # Add green ball with gradient and texture
    image = generate_green_ball(width, height, image)
    return image


def grid_sample(tensor: torch.Tensor, grid: torch.Tensor) -> torch.Tensor:
    """Given an input and a flow-field grid, computes the output using input
    values and pixel locations from grid.

    Args:
        tensor: (N, C, H_in, W_in) tensor
        grid: (N, H_out, W_out, 2) tensor in the range of [-1, 1]

    Returns:
        (N, C, H_out, W_out) tensor

    See `torch.nn.functional.grid_sample`.
    """
    b, c, h, w = tensor.shape
    b_, h_out, w_out, w_ = grid.shape
    assert b == b_ and w_ == 2
    out = []
    for (t, g) in zip(tensor, grid):
        x_ = 0.5 * (w - 1) * (g[..., 0].reshape(-1) + 1)
        y_ = 0.5 * (h - 1) * (g[..., 1].reshape(-1) + 1)
        ix = torch.floor(x_).clamp(0, w - 2).long()
        iy = torch.floor(y_).clamp(0, h - 2).long()
        dx = x_ - ix
        dy = y_ - iy
        out.append((1 - dx) * (1 - dy) * t[..., iy, ix]
                   + dx * (1 - dy) * t[..., iy, ix + 1]
                   + (1 - dx) * dy * t[..., iy + 1, ix]
                   + dx * dy * t[..., iy + 1, ix + 1])
    return torch.cat(out, dim=0).reshape(b, c, h_out, w_out)


def create_scale_grid(reference_height, reference_width, grid_height, grid_width):
    grid = torch.dstack(torch.meshgrid(torch.linspace(0, reference_width - 1, grid_width, dtype=torch.float64),
                                       torch.linspace(0, reference_height - 1, grid_height, dtype=torch.float64),
                                       indexing='xy'))
    grid = grid[None, ...]
    return grid


# create absulut grid
def generate_absolut_grid(
    grid,
    grid_width: int,
    grid_height: int,
    grid_bit_width: int,

):
    loc_grid = torch.dstack(torch.meshgrid(torch.arange(grid_width), torch.arange(grid_height), indexing='xy'))

    delta_grid = grid - loc_grid
    max_motion = torch.ceil(torch.abs(delta_grid.reshape((-1, 2))).max(dim=0).values)
    frac_bits = grid_bit_width - torch.floor(torch.log2(max_motion)) - 2
    delta_grid = torch.round(delta_grid * frac_bits.exp2()).to(torch.int16)
    delta_grid = delta_grid.permute(2, 0, 1)[None, ...]

    # height, width = delta_grid.shape[2:]
    # xs_mesh, ys_mesh = torch.meshgrid(torch.arange(width), torch.arange(height), indexing='xy')
    # delta_grid[0, 0, ...] += xs_mesh 
    # delta_grid[0, 1, ...] += ys_mesh
    
    return delta_grid, max_motion, frac_bits.reshape(1, 2, 1, 1)

def grid_quantization(
    grid,
    grid_bit_width: int,

):

    delta_grid = grid
    max_motion = torch.ceil(torch.abs(delta_grid.reshape((-1, 2))).max(dim=0).values)
    frac_bits = grid_bit_width - torch.floor(torch.log2(max_motion)) - 2
    # TODO add assertions to avoid 0 frac bits and more than 8 frac bits
    delta_grid = torch.round(delta_grid * frac_bits.exp2()).to(torch.int16)
    delta_grid = delta_grid.permute(2, 0, 1)[None, ...]
    
    return delta_grid, max_motion, frac_bits.reshape(1, 2, 1, 1)


def split_msb_lsb(q: torch.Tensor, frac_bits):
    """Extract the msb/lsb split of a fixed-point integer grid.

        idx  = q >> frac_bits          # msb: reference index (floor, works for negatives)
        frac = q & (2**frac_bits - 1)  # lsb: fraction, always in [0, 2**frac_bits)

    Guarantees  q == idx * 2**frac_bits + frac  also for signed q
    (arithmetic shift + two's-complement mask).

    frac_bits can be a scalar or broadcastable to q (e.g. per-axis [1, 2, 1, 1]).
    Returns (idx, frac) as long tensors.
    """
    if not q.dtype in (torch.int16, torch.int32, torch.int64):
        raise TypeError(f"expected an integer grid, got {q.dtype}")
    fr = torch.as_tensor(frac_bits).long()
    idx = q.long() >> fr
    frac = q.long() & ((torch.ones_like(fr) << fr) - 1)
    return idx, frac


def torch_warp(src, warp_grid):
    """Torch version of `np_warp`: bilinear warp on float operands."""
    height, width = warp_grid.shape[2:]
    src_height, src_width = src.shape[2:]
    xs, ys = torch.meshgrid(torch.arange(width), torch.arange(height), indexing='xy')
    xs = xs + warp_grid[0, 0, ...]
    ys = ys + warp_grid[0, 1, ...]
    xs0 = torch.clamp(torch.floor(xs), 0, src_width - 2).long()
    ys0 = torch.clamp(torch.floor(ys), 0, src_height - 2).long()

    w00 = torch.clamp(xs0 + 1 - xs, 0, 1) * torch.clamp(ys0 + 1 - ys, 0, 1)
    w10 = torch.clamp(xs0 + 1 - xs, 0, 1) * torch.clamp(ys - ys0, 0, 1)
    w01 = torch.clamp(xs - xs0, 0, 1) * torch.clamp(ys0 + 1 - ys, 0, 1)
    w11 = torch.clamp(xs - xs0, 0, 1) * torch.clamp(ys - ys0, 0, 1)

    out00 = src[..., ys0, xs0] * w00[None, None, ...]
    out01 = src[..., ys0, xs0 + 1] * w01[None, None, ...]
    out10 = src[..., ys0 + 1, xs0] * w10[None, None, ...]
    out11 = src[..., ys0 + 1, xs0 + 1] * w11[None, None, ...]

    out = out00 + out01 + out10 + out11

    return out


def infer_warp(graph, node):
    blobs_data = graph.get_node_blobs_data(node)
    warp_grid = blobs_data['_input'].clone()
    reference = blobs_data['_warp._reference']
    # calculate fractional bits from limit - need to take from hw policy
    # e.g. grid type int16, limit 127 need 7 bits for int part, 8 bits for fraction (this is per axes x/y)
    grid_node = graph.get_blob(node, '_input')
    mantissa = grid_node.format.mantissa
    pixel_bit_limit = torch.log2(torch.tensor([float(node.warp.limit.x), float(node.warp.limit.y)], dtype=torch.float64))
    pixel_bit_limit = torch.floor(pixel_bit_limit + 1.)
    min_frac_bit_limit = torch.full((2,), float((mantissa + int(grid_node.format.sign)) // 2), dtype=torch.float64)
    frac_bits = torch.min(torch.stack((mantissa - pixel_bit_limit, min_frac_bit_limit)), dim=0).values.reshape(1, 2, 1, 1)
    assert bool(torch.all(frac_bits >= 0)), 'number of fraction bits in warp-grid is not supported for negative values'
    float_warp_grid = warp_grid.double() * (-frac_bits).exp2()
    float_reference = reference.double()
    offset = node.warp.reference_offset or Dimensions(x=0, y=0, z=0, n=0)
    offset = torch.tensor([offset.x, offset.y], dtype=torch.float64).reshape([1, -1, 1, 1])
    limit = torch.tensor([node.warp.limit.x, node.warp.limit.y], dtype=torch.float64).reshape([1, -1, 1, 1])
    float_warp_grid = torch.clamp(float_warp_grid, -limit, limit) + offset
    out_data = torch_warp(float_reference, float_warp_grid)

    # This multiplication is due to the fact that warp implementation has bilinear interpolation
    # Which comes down to multiplication of two numbers with fl=frac_bits each. In order to
    # return to non float number we need to multiply the result in 2 ** (2 * frac_bits)
    # Same as we do when we force bias delta to be the multiplication of ker and input deltas
    # We could have use float_reference as reference * 2. ** -reference_frac_bits and then
    # we would do out_data = (out_data * 2 ** (2 * frac_bits * ref_frac_bits)).astype(np.uint16)
    out_data = out_data * frac_bits.sum().exp2()
    assert bool(torch.equal(out_data, out_data.to(torch.int32).double()))
    out_data = out_data.to(torch.int32)
    out_blob = graph.get_blob(node, '_output')
    graph.set_blob_data(out_blob, out_data)
    graph.log.debug('inferred warp node: %s' % node.name)

## Work area — torch translation of tabs 11–22

In [22]:
# calibrate the reference: integer-input observer rule (raw ints, int8 range-fit)
obs = MXObserver(mode="input")

In [23]:
reference_height, reference_width, grid_height, grid_width = 240, 320, 80, 106
grid_bits = 16
x = generate_image(reference_height, reference_width).permute(2, 0, 1).unsqueeze(0)
grid = create_scale_grid(reference_height, reference_width, grid_height, grid_width)
new_h, new_w = grid.shape[-3:-1]
orig_h, orig_w = x.shape[-2:]

# get expected output
grid_norm_scale = 2 / torch.tensor([orig_w - 1, orig_h - 1], dtype=torch.float64).reshape((1, 1, 1, -1))
grid_normed = grid_norm_scale * grid - 1
expected = grid_sample(x, grid_normed)

# generate relative hardware grid
hw_grid, max_motion, frac_bits = generate_absolut_grid(
    grid[0],
    new_w,
    new_h,
    grid_bits
)

# create graph for inference engine
warp_props = NodeWarp()
warp_props.limit = Dimensions(x=float(max_motion[0]), y=float(max_motion[1]))
node = Node(name='warper', warp=warp_props)
blobs_data = {
    '_input': hw_grid,
    '_warp._reference': x.clone(),
    '_output': torch.empty(expected.shape, dtype=torch.int32)
}

blobs = {
    '_input': Blob(name='grid', format=Format(sign=True, exponent=0, mantissa=15)),
    '_output': Blob(name='output', format=Format(sign=True, exponent=0, mantissa=31))
}

graph = MockGraph(blobs_data, blobs)

# run inference
infer_warp(graph, node)

# descale and compare results
result = blobs_data['_output'].reshape(expected.shape)
result = result / frac_bits.sum().exp2()


In [24]:
quantized_grid, max_motion, frac_bits = grid_quantization(
    grid[0],
    grid_bits
)

src = x.clone()


In [25]:
x_offset, x_frac = split_msb_lsb(quantized_grid[0, 0, ...].to(dtype=torch.int16), frac_bits[0][0])
y_offset, y_frac = split_msb_lsb(quantized_grid[0, 1, ...].to(dtype=torch.int16), frac_bits[0][1])

In [26]:
# get interpulation weights
# w00 = torch.clamp(xs0 + 1 - xs, 0, 1) * torch.clamp(ys0 + 1 - ys, 0, 1)
# w10 = torch.clamp(xs0 + 1 - xs, 0, 1) * torch.clamp(ys - ys0, 0, 1)
# w01 = torch.clamp(xs - xs0, 0, 1) * torch.clamp(ys0 + 1 - ys, 0, 1)
# w11 = torch.clamp(xs - xs0, 0, 1) * torch.clamp(ys - ys0, 0, 1)

# get interpulation weights
w00 = (2 ** frac_bits[0][0] - x_frac) * (2 ** frac_bits[0][1] - y_frac)
w10 = (2 ** frac_bits[0][0] - x_frac) * (y_frac)
w01 = (x_frac) * (2 ** frac_bits[0][1] - y_frac)
w11 = (x_frac) * (y_frac)


In [27]:
# get reference indices
# ref00 = src[..., ys0, xs0]
# ref01 = src[..., ys0, xs0 + 1]
# ref10 = src[..., ys0 + 1, xs0]
# ref11 = src[..., ys0 + 1, xs0 + 1]

# get reference indices
src_height, src_width = src.shape[2:]

xs0 = torch.clamp(torch.floor(x_offset), 0, src_width - 2).long()
ys0 = torch.clamp(torch.floor(y_offset), 0, src_height - 2).long()

ref00 = src[..., ys0, xs0]
ref01 = src[..., ys0, xs0 + 1]
ref10 = src[..., ys0 + 1, xs0]
ref11 = src[..., ys0 + 1, xs0 + 1]



In [28]:
# bcu computation - it is an elementwise computation (each time one element)
out00 = ref00 * w00[None, None, ...]
out01 = ref01 * w01[None, None, ...]
out10 = ref10 * w10[None, None, ...]
out11 = ref11 * w11[None, None, ...]

out = out00 + out01 + out10 + out11

In [29]:
q_res = out / 2. ** frac_bits.sum()


In [30]:
name = "downscale"
print(f'[{name}] mse q_expected vs hw result: {((expected - result) ** 2).mean().item():.3f}')
print(f'[{name}] mse q_expected vs actual hw results: {((expected - q_res) ** 2).mean().item():.3f}')

[downscale] mse q_expected vs hw result: 0.002
[downscale] mse q_expected vs actual hw results: 0.007


### HARADWARE FUNCTIONS IMPLEMENTATION

In [31]:
def gen_warp_weights(x_frac, y_frac, frac_bits):
    w00 = (2 ** frac_bits[0][0] - x_frac) * (2 ** frac_bits[0][1] - y_frac)
    w10 = (2 ** frac_bits[0][0] - x_frac) * (y_frac)
    w01 = (x_frac) * (2 ** frac_bits[0][1] - y_frac)
    w11 = (x_frac) * (y_frac)

    return w00, w01, w10, w11

In [32]:
def get_ref_frame(src, x_offset, y_offset):
    # this is a relative grid and to be able to handle the last element in the line it copied
    src_height, src_width = src.shape[2:]

    xs0 = torch.clamp(torch.floor(x_offset), 0, src_width - 2).long()
    ys0 = torch.clamp(torch.floor(y_offset), 0, src_height - 2).long()

    ref00 = src[..., ys0, xs0]
    ref01 = src[..., ys0, xs0 + 1]
    ref10 = src[..., ys0 + 1, xs0]
    ref11 = src[..., ys0 + 1, xs0 + 1]

    return ref00, ref01, ref10, ref11

In [33]:
# new warp: implement the TRC here
def trc(quantized_grid, frac_bits):
    x_offset, x_frac = split_msb_lsb(quantized_grid[0, 0, ...].to(dtype=torch.int16), frac_bits[0][0])
    y_offset, y_frac = split_msb_lsb(quantized_grid[0, 1, ...].to(dtype=torch.int16), frac_bits[0][1])
    # interpulate_weights = torch.stack(gen_warp_weights(x_frac, y_frac))
    # frame_coords = torch.stack(get_ref_frame(src, ys0, xs0))

    tru_a = gen_warp_weights(x_frac, y_frac, frac_bits)
    tru_w = get_ref_frame(src, x_offset, y_offset)

    return tru_a, tru_w

In [34]:
def to_calculate_in_bcu(tru_a, tru_w):
    w00, w01, w10, w11= tru_a
    ref00, ref01, ref10, ref11 = tru_w
    out00 = ref00 * w00[None, None, ...]
    out01 = ref01 * w01[None, None, ...]
    out10 = ref10 * w10[None, None, ...]
    out11 = ref11 * w11[None, None, ...]

    out = out00 + out01 + out10 + out11
    return out 

In [35]:
def dequant_warp_res(out, frac_bits):
    return out / 2. ** frac_bits.sum()

In [36]:
quantized_grid, max_motion, frac_bits = grid_quantization(
    grid[0],
    grid_bits
)

src = x.clone()
tru_a, tru_w = trc(quantized_grid, frac_bits)
out2 = to_calculate_in_bcu(tru_a, tru_w)
q_res = dequant_warp_res(out2, frac_bits)
print(f'[{name}] mse q_expected vs actual hw results: {((expected - q_res) ** 2).mean().item():.3f}')

[downscale] mse q_expected vs actual hw results: 0.007
